In [ ]:
# Start

In [11]:
# Import Packages, load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

items = pd.read_csv("OneDrive/FHNW/8. Semester/Analytics Project/items.csv", sep="|")
train = pd.read_csv("OneDrive/FHNW/8. Semester/Analytics Project/train.csv", sep="|")

In [15]:
# Check data
items.head()
train.head()


,lineID,day,pid,adFlag,availability,competitorPrice,click,basket,order,price,revenue
0,1,1,6570,0,2,14.60,1.0,0.0,0.0,16.89,0.00
1,2,1,14922,1,1,8.57,0.0,1.0,0.0,8.75,0.00
2,3,1,16382,0,1,14.77,0.0,1.0,0.0,16.06,0.00
3,4,1,1145,1,1,6.59,0.0,0.0,1.0,6.55,6.55
4,5,1,3394,0,1,4.39,0.0,0.0,1.0,4.14,4.14


In [13]:
# Check data
items.info()
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22035 entries, 0 to 22034
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pid             22035 non-null  int64  
 1   manufacturer    22035 non-null  int64  
 2   group           22035 non-null  object 
 3   content         22035 non-null  object 
 4   unit            22035 non-null  object 
 5   pharmForm       19708 non-null  object 
 6   genericProduct  22035 non-null  int64  
 7   salesIndex      22035 non-null  int64  
 8   category        17408 non-null  float64
 9   campaignIndex   1338 non-null   object 
 10  rrp             22035 non-null  float64
dtypes: float64(2), int64(4), object(5)
memory usage: 1.8+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28406 entries, 0 to 28405
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   lineID           28406 non-null  int64

In [59]:
[
    (df["price"] < 0).sum(),
    (df["competitorPrice"] < 0).sum(),
    df["price"].describe()
]

[np.int64(0),
 np.int64(0),
 count    28405.000000
 mean        12.965136
 std         12.382542
 min          0.080000
 25%          5.450000
 50%          9.850000
 75%         15.350000
 max        262.960000
 Name: price, dtype: float64]

In [14]:
df = train.merge(items, on="pid", how="left")
df.head()

,lineID,day,pid,adFlag,availability,competitorPrice,click,basket,order,price,revenue,manufacturer,group,content,unit,pharmForm,genericProduct,salesIndex,category,campaignIndex,rrp
0,1,1,6570,0,2,14.60,1.0,0.0,0.0,16.89,0.00,255,2FOI,50,ML,TRO,0,40,193.0,NaN,18.25
1,2,1,14922,1,1,8.57,0.0,1.0,0.0,8.75,0.00,18,1COJ0FIK,50,ST,TAB,1,40,66.0,C,18.81
2,3,1,16382,0,1,14.77,0.0,1.0,0.0,16.06,0.00,41,22OI7,2X50,ML,STI,0,53,40.0,NaN,18.48
3,4,1,1145,1,1,6.59,0.0,0.0,1.0,6.55,6.55,52,18OZ00IS,60,G,GEL,0,40,25.0,NaN,9.31
4,5,1,3394,0,1,4.39,0.0,0.0,1.0,4.14,4.14,90,20OI0,25X2,ST,KOM,0,53,14.0,NaN,8.13


In [16]:
# Check missing values
df.isna().mean().sort_values(ascending=False)

## Values near 1.0 = almost completely missing → maybe drop
## Values 0.1 to 0.5 = moderate → consider imputation
## Values < 0.05 = light → easy to fill

campaignIndex      0.785503
pharmForm          0.064986
competitorPrice    0.044850
category           0.032282
revenue            0.000035
click              0.000035
basket             0.000035
order              0.000035
price              0.000035
content            0.000000
salesIndex         0.000000
genericProduct     0.000000
unit               0.000000
lineID             0.000000
group              0.000000
manufacturer       0.000000
day                0.000000
availability       0.000000
adFlag             0.000000
pid                0.000000
rrp                0.000000
dtype: float64

In [17]:
# Clean code
# 1. campaignIndex: Replace NaN with "NONE"
df["campaignIndex"] = df["campaignIndex"].fillna("NONE")

# Convert to string if not already
df["campaignIndex"] = df["campaignIndex"].astype(str)


# 2. pharmForm: Fill missing with "UNKNOWN"
df["pharmForm"] = df["pharmForm"].fillna("UNKNOWN")


# 3. category: Replace missing with -1 or "UNKNOWN"
df["category"] = df["category"].fillna(-1)


# 4. competitorPrice: fill missing competitor prices
# Option A: median competitor price per product group
df["competitorPrice"] = df.groupby("group")["competitorPrice"].transform(
    lambda x: x.fillna(x.median())
)

# Option B fallback, in case entire group is missing
df["competitorPrice"] = df["competitorPrice"].fillna(df["competitorPrice"].median())

In [19]:
# Check values if cleaned
df.isna().mean().sort_values(ascending=False)

# result=acceptable


revenue            0.000035
click              0.000035
basket             0.000035
order              0.000035
price              0.000035
content            0.000000
campaignIndex      0.000000
category           0.000000
salesIndex         0.000000
genericProduct     0.000000
pharmForm          0.000000
unit               0.000000
lineID             0.000000
group              0.000000
manufacturer       0.000000
day                0.000000
competitorPrice    0.000000
availability       0.000000
adFlag             0.000000
pid                0.000000
rrp                0.000000
dtype: float64

In [ ]:
# Feature Generation

In [21]:
# Create price based Features

# 1. absolute price difference to competitor
df["price_diff"] = df["price"] - df["competitorPrice"]

# 2. percentage difference (elasticity indicator)
df["price_ratio"] = df["price"] / df["competitorPrice"]

# 3. discount relative to reference price (manufacturer recommended price)
df["discount_rrp"] = (df["rrp"] - df["price"]) / df["rrp"]

# 4. log prices (good for ML models)
df["log_price"] = np.log1p(df["price"])
df["log_competitorPrice"] = np.log1p(df["competitorPrice"])

In [23]:
# Meaning of availability 1,2,3,4

df.groupby("availability")["order"].agg(["count", "mean"])

## 1: highest order rate + highest count, assume product is stocked
## 2: Lower order rate / count, assume low stock
## 3: Lowest order rate / count, assume critically low stock
## 4: Non-existent within the dataset, assume no orders due to no stock


,count,mean
availability,,
1,26391,0.357698
2,1636,0.228606
3,378,0.201058


In [25]:
# Create availability Features

availability_map = {
    1: "in_stock",
    2: "low_stock",
    3: "critical_stock",
    4: "out_of_stock"
}

df["availability_status"] = df["availability"].map(availability_map)
df["is_out_of_stock"] = (df["availability"] == 4).astype(int)

In [44]:
# Create campaign Feature
## Q: Does advertising influence orders?

# Campaign active or not
df["has_campaign"] = (df["campaignIndex"] != "NONE").astype(int)

# Advertising flag (already 0/1 but we rename for clarity)
df["is_advertised"] = df["adFlag"].astype(int)


In [27]:
# Rename genericProduct
# Create Feature to allow impact is_generic and discount_rrp

df["is_generic"] = df["genericProduct"]

df["generic_discount_interaction"] = df["discount_rrp"] * df["is_generic"]

In [28]:
# Product Group + Manufacturer Features

df["group_manufacturer"] = df["group"].astype(str) + "_" + df["manufacturer"].astype(str)

In [29]:
# Check Content Feature

df["content"].astype(str).unique()[:50]

array(['50', '2X50', '60', '25X2', '1000', '20', '100', '1', '3', '6',
       '10', '30', '12', '500', '400', '150', '80', '75', '12X10', '125',
       '35', '200', '25', '10X10', '90', '30X0.5', '3X10', '350', '15',
       '14', '84', '750', '5', '4X0.8', '430', '1X50', '10X1', '120', '2',
       '450', '28', '4', '3X3', '180', '250', '24', '40', '6X4X200',
       '160', '600'], dtype=object)

In [30]:
# Create interpretation of Feature content
import numpy as np

def parse_content(value):
    """
    Parse the 'content' column into a numeric total volume.
    Handles formats like:
      - "50"
      - "2X50"
      - "30X0.5"
      - "6X4X200"
      - floats or ints
    """
    # Convert to string
    val = str(value)

    # If it contains "X", treat it as a multiplier expression
    if "X" in val:
        try:
            parts = val.split("X")
            numbers = [float(p) for p in parts]
            total = np.prod(numbers)  # multiply all parts
            return total
        except:
            return np.nan

    # Otherwise: try to parse as a simple float
    try:
        return float(val)
    except:
        return np.nan

In [31]:
# Apply

df["content_volume"] = df["content"].apply(parse_content)

In [32]:
# Check

df[["content", "content_volume"]].head(20)

## Check=Noice

,content,content_volume
0,50,50.0
1,50,50.0
2,2X50,100.0
3,60,60.0
4,25X2,50.0
5,1000,1000.0
6,20,20.0
7,100,100.0
8,50,50.0
9,100,100.0


In [36]:
# Rewind aggregation Features ERROR
# Remove old aggregated columns if they already exist
cols_to_remove = [
    "pid_order_rate",
    "pid_basket_rate",
    "pid_click_rate",
    "pid_avg_price",
    "pid_avg_competitorPrice"
]

df = df.drop(columns=[c for c in cols_to_remove if c in df.columns])

In [37]:
# Create aggregation Features for each PID
# 1. Aggregate behavioral + pricing signals at product (pid) level
product_stats = (
    df.groupby("pid")
      .agg({
          "order": "mean",               # overall purchase probability of this product
          "basket": "mean",              # how often it's added to basket
          "click": "mean",               # how often it's clicked
          "price": "mean",               # typical product price
          "competitorPrice": "mean"      # typical competitor pricing
      })
      .rename(columns={
          "order": "pid_order_rate",
          "basket": "pid_basket_rate",
          "click": "pid_click_rate",
          "price": "pid_avg_price",
          "competitorPrice": "pid_avg_competitorPrice"
      })
)

# 2. Join these aggregated values back to the main dataframe
df = df.join(product_stats, on="pid")

# 3. Clean any missing values in the new aggregated columns (rare edge cases)
agg_cols = [
    "pid_order_rate",
    "pid_basket_rate",
    "pid_click_rate",
    "pid_avg_price",
    "pid_avg_competitorPrice"
]

for col in agg_cols:
    df[col] = df[col].fillna(df[col].mean())


# 4. Sanity check: show first rows of the new aggregated features
df[["pid",
    "pid_order_rate",
    "pid_basket_rate",
    "pid_click_rate",
    "pid_avg_price",
    "pid_avg_competitorPrice"]].head()

,pid,pid_order_rate,pid_basket_rate,pid_click_rate,pid_avg_price,pid_avg_competitorPrice
0,6570,0.000000,0.500000,0.500000,16.890,14.600000
1,14922,0.517241,0.172414,0.310345,8.750,8.570000
2,16382,0.500000,0.500000,0.000000,15.915,14.770000
3,1145,0.373333,0.146667,0.480000,6.550,6.323333
4,3394,0.571429,0.142857,0.285714,4.140,4.924286


In [39]:
# Check if days can be used and assigned a weekday

df["day"].min(), df["day"].max()
df["day"].value_counts().sort_index()

# Feature day not usable!

day
1    18358
2    10048
Name: count, dtype: int64

In [46]:
"competitor_price_change" in df.columns

False

In [47]:
# ---------------------------------------------------
# STEP 8 (Final Correct Version For Your Dataset)
# ---------------------------------------------------

# Ensure 'day' is integer
df["day"] = df["day"].astype(int)

# Binary flag for Day 2
df["is_day2"] = (df["day"] == 2).astype(int)

# Price change per product (Day 2 - Day 1)
df["price_change"] = df.groupby("pid")["price"].diff().fillna(0)

# Competitor price change per product
df["competitor_price_change"] = df.groupby("pid")["competitorPrice"].diff().fillna(0)

# Availability change per product
df["availability_change"] = df.groupby("pid")["availability"].diff().fillna(0)

In [48]:
df[["price_change", "competitor_price_change", "availability_change"]].head()

,price_change,competitor_price_change,availability_change
0,0.0,0.0,0.0
1,0.0,0.0,0.0
2,0.0,0.0,0.0
3,0.0,0.0,0.0
4,0.0,0.0,0.0


In [53]:
[
    train["price"].nunique(),
    train["competitorPrice"].nunique(),
    train["availability"].nunique()
]

[1970, 2374, 3]

In [50]:
df.groupby(["pid", "day"])["price"].nunique().head(20)

pid  day
3    1      1
4    2      1
8    1      1
10   1      1
     2      1
11   1      1
     2      1
13   1      1
14   1      1
15   1      1
     2      1
18   2      1
24   1      1
28   1      1
30   1      1
32   2      1
34   2      1
38   1      1
     2      1
43   2      1
Name: price, dtype: int64

In [54]:
# ---------------------------------------------------
# REMOVE UNWANTED STEP 8 FEATURES
# ---------------------------------------------------

cols_to_remove = [
    "is_day2",
    "price_change",
    "competitor_price_change",
    "availability_change",
    "pid_day_click_rate",
    "pid_day_basket_rate",
    "pid_day_order_rate"
]

# Remove only columns that exist (avoid errors)
df = df.drop(columns=[c for c in cols_to_remove if c in df.columns], errors="ignore")

In [55]:
df.columns.tolist()


['lineID',
 'day',
 'pid',
 'adFlag',
 'availability',
 'competitorPrice',
 'click',
 'basket',
 'order',
 'price',
 'revenue',
 'manufacturer',
 'group',
 'content',
 'unit',
 'pharmForm',
 'genericProduct',
 'salesIndex',
 'category',
 'campaignIndex',
 'rrp',
 'price_diff',
 'price_ratio',
 'discount_rrp',
 'log_price',
 'log_competitorPrice',
 'availability_status',
 'is_out_of_stock',
 'is_generic',
 'generic_discount_interaction',
 'group_manufacturer',
 'content_volume',
 'pid_order_rate',
 'pid_basket_rate',
 'pid_click_rate',
 'pid_avg_price',
 'pid_avg_competitorPrice',
 'has_campaign',
 'is_advertised',
 'ad_price_diff',
 'generic_discount',
 'campaign_price_diff',
 'content_price_ratio',
 'click_x_day',
 'generic_price_diff']

In [56]:
df.head(20)

,lineID,day,pid,adFlag,availability,competitorPrice,click,basket,order,price,revenue,manufacturer,group,content,unit,pharmForm,genericProduct,salesIndex,category,campaignIndex,rrp,price_diff,price_ratio,discount_rrp,log_price,log_competitorPrice,availability_status,is_out_of_stock,is_generic,generic_discount_interaction,group_manufacturer,content_volume,pid_order_rate,pid_basket_rate,pid_click_rate,pid_avg_price,pid_avg_competitorPrice,has_campaign,is_advertised,ad_price_diff,generic_discount,campaign_price_diff,content_price_ratio,click_x_day,generic_price_diff
0,1,1,6570,0,2,14.60,1.0,0.0,0.0,16.89,0.00,255,2FOI,50,ML,TRO,0,40,193.0,NONE,18.25,2.29,1.156849,0.074521,2.884242,2.747271,low_stock,0,0,0.000000,2FOI_255,50.0,0.000000,0.500000,0.500000,16.890000,14.600000,0,0,0.00,0.000000,0.00,2.960332,0.500000,0.00
1,2,1,14922,1,1,8.57,0.0,1.0,0.0,8.75,0.00,18,1COJ0FIK,50,ST,TAB,1,40,66.0,C,18.81,0.18,1.021004,0.534822,2.277267,2.258633,in_stock,0,1,0.534822,1COJ0FIK_18,50.0,0.517241,0.172414,0.310345,8.750000,8.570000,1,1,0.18,0.534822,0.18,5.714286,0.310345,0.18
2,3,1,16382,0,1,14.77,0.0,1.0,0.0,16.06,0.00,41,22OI7,2X50,ML,STI,0,53,40.0,NONE,18.48,1.29,1.087339,0.130952,2.836737,2.758109,in_stock,0,0,0.000000,22OI7_41,100.0,0.500000,0.500000,0.000000,15.915000,14.770000,0,0,0.00,0.000000,0.00,6.226650,0.000000,0.00
3,4,1,1145,1,1,6.59,0.0,0.0,1.0,6.55,6.55,52,18OZ00IS,60,G,GEL,0,40,25.0,NONE,9.31,-0.04,0.993930,0.296455,2.021548,2.026832,in_stock,0,0,0.000000,18OZ00IS_52,60.0,0.373333,0.146667,0.480000,6.550000,6.323333,0,1,-0.04,0.000000,-0.00,9.160305,0.480000,-0.00
4,5,1,3394,0,1,4.39,0.0,0.0,1.0,4.14,4.14,90,20OI0,25X2,ST,KOM,0,53,14.0,NONE,8.13,-0.25,0.943052,0.490775,1.637053,1.684545,in_stock,0,0,0.000000,20OI0_90,50.0,0.571429,0.142857,0.285714,4.140000,4.924286,0,0,-0.00,0.000000,-0.00,12.077295,0.285714,-0.00
5,6,1,3661,0,1,13.66,0.0,0.0,1.0,10.03,10.03,90,13OX06,1000,ML,LOE,0,52,127.0,NONE,21.60,-3.63,0.734261,0.535648,2.400619,2.685123,in_stock,0,0,0.000000,13OX06_90,1000.0,0.428571,0.428571,0.142857,10.030000,12.894286,0,0,-0.00,0.000000,-0.00,99.700897,0.142857,-0.00
6,7,1,3856,1,1,3.03,0.0,0.0,1.0,3.58,3.58,84,13OK0FOK,20,G,SAL,0,40,90.0,NONE,5.62,0.55,1.181518,0.362989,1.521699,1.393766,in_stock,0,0,0.000000,13OK0FOK_84,20.0,0.581818,0.272727,0.145455,3.580000,3.030000,0,1,0.55,0.000000,0.00,5.586592,0.145455,0.00
7,8,1,16963,0,1,8.78,1.0,0.0,0.0,8.75,0.00,89,22OI3,100,ML,SAL,0,53,8.0,NONE,11.62,-0.03,0.996583,0.246988,2.277267,2.280339,in_stock,0,0,0.000000,22OI3_89,100.0,0.428571,0.285714,0.285714,8.750000,8.218571,0,0,-0.00,0.000000,-0.00,11.428571,0.285714,-0.00
8,9,1,14560,0,1,10.84,1.0,0.0,0.0,12.04,0.00,891,22OI3,50,ML,CRE,0,53,8.0,NONE,14.19,1.20,1.110701,0.151515,2.568022,2.471484,in_stock,0,0,0.000000,22OI3_891,50.0,0.200000,0.200000,0.600000,12.040000,10.786000,0,0,0.00,0.000000,0.00,4.152824,0.600000,0.00
9,10,1,4853,1,1,9.12,1.0,0.0,0.0,8.75,0.00,590,13OX04OZ,100,G,SAL,0,40,90.0,NONE,14.25,-0.37,0.959430,0.385965,2.277267,2.314514,in_stock,0,0,0.000000,13OX04OZ_590,100.0,0.142857,0.285714,0.571429,8.750000,9.120000,0,1,-0.37,0.000000,-0.00,11.428571,0.571429,-0.00


"We engineered pricing, behavioral, marketing, and product-specific features to improve the predictive power of the model.
Pricing features (price_diff, discount_rrp) quantify competitiveness and discounting.
Behavioral aggregates (pid_order_rate, pid_click_rate) capture product popularity from historical interactions.
Content parsing extracts true package size, which affects value perception.
Marketing features (is_advertised, has_campaign) measure promotion effects.
Interaction features model non-linear relationships such as how discount impacts generics or how content size interacts with pricing.
Together, these features enable a richer representation of product-level dynamics that influence click, basket, and purchase behaviors."

In [ ]:
Delta Vortag, Vorwoche
